In [6]:
from google.colab import drive
from pathlib import Path
import sys
import os

drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = Path('/content/drive/Othercomputers/My Laptop/thesis_project')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)

print("Working directory:", os.getcwd())

Mounted at /content/drive
Working directory: /content/drive/Othercomputers/My Laptop/thesis_project


In [ ]:
from python.src.train.model_trainer import train_model
from pathlib import Path
import torch

PROJECT_ROOT = Path.cwd()

models = ["asymmetric_trinet"]
version = "v2"
seeds = [38]
n_per_class = [2500]
epochs = 30

for m in models:
  for s in seeds:
    for n in n_per_class:

      print(f"\n\nRunning experiment model = {m}, seed = {s}, n_per_class = {n}")
      print("============================================================================\n")

      trained_model = train_model(seed=s,
                  project_root=PROJECT_ROOT,
                  model_name=m,
                  n_per_class=n,
                  spec_version=version,
                  n_epochs=epochs
                  )
      # clean up RAM
      del trained_model
      if torch.cuda.is_available():
            torch.cuda.empty_cache()
      elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            torch.mps.empty_cache()
      print("\nTraining finished")



Running experiment model = asymmetric_trinet, seed = 38, n_per_class = 2500


Validation report found: /content/drive/Othercomputers/My Laptop/thesis_project/reports/validations/validation_seed38_n2500_v2.json
Starting training...

Using device: cuda

Epoch 01 | LR: 0.001000 | Train: 1.0182 (CE 0.8661 / SupCon 1.5210) | Val Loss: 0.7484 | Val Acc: 92.40
Epoch 05 | LR: 0.000957 | Train: 0.7078 (CE 0.5910 / SupCon 1.1686) | Val Loss: 0.5510 | Val Acc: 99.98
Epoch 10 | LR: 0.000794 | Train: 0.6577 (CE 0.5461 / SupCon 1.1154) | Val Loss: 0.5657 | Val Acc: 98.62
Epoch 15 | LR: 0.000552 | Train: 0.6438 (CE 0.5341 / SupCon 1.0979) | Val Loss: 0.5381 | Val Acc: 99.96
Epoch 20 | LR: 0.000297 | Train: 0.6313 (CE 0.5236 / SupCon 1.0764) | Val Loss: 0.5381 | Val Acc: 99.98
Epoch 25 | LR: 0.000095 | Train: 0.6180 (CE 0.5127 / SupCon 1.0526) | Val Loss: 0.5353 | Val Acc: 99.98
Epoch 30 | LR: 0.000003 | Train: 0.6142 (CE 0.5097 / SupCon 1.0457) | Val Loss: 0.5356 | Val Acc: 99.98

Model saved to: /

In [ ]:
from python.src.train.osr_trainer import train_osr_model
from python.src.train.osr_hparams import OSRHParams
from pathlib import Path
import torch

PROJECT_ROOT = Path.cwd()

version = "v2"
seeds = [38]
n_per_class = [2500]
epochs = 25

# Initialize hyperparameters (tweak these directly here if needed)
hparams = OSRHParams()

for s in seeds:
    for n in n_per_class:

        print(f"\n\nRunning OSR experiment | seed = {s}, n_per_class = {n}")
        print("============================================================================\n")

        trained_model = train_osr_model(
            seed=s,
            n_per_class=n,
            spec_version=version,
            project_root=PROJECT_ROOT,
            epochs=epochs,
            hparams=hparams
        )

        # clean up RAM
        del trained_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            torch.mps.empty_cache()

        print("Trainng Complete")



Running OSR experiment | seed = 38, n_per_class = 2500


OsrSAF_TriNet | seed=38 | n=2500
Device              : cuda
Closed-set ckpt     : asymmetric_trinet_seed38_n2500.pt
Codebook fill epochs: 5
Phase 2 (calibrator): until epoch 25
Target FPR          : 0.10

[OsrSAF_TriNet] Loaded backbone from /content/drive/Othercomputers/My Laptop/thesis_project/artifacts/checkpoints/asymmetric_trinet_seed38_n2500.pt

[Stage 2.A] Populating codebook over 5 epochs (frozen backbone)

  Fill epoch 1/5 | init=100% | spread=0.0025 | updates/centroid=386.6
  Fill epoch 2/5 | init=100% | spread=0.0027 | updates/centroid=773.5
  Fill epoch 3/5 | init=100% | spread=0.0029 | updates/centroid=1161.2
  Fill epoch 4/5 | init=100% | spread=0.0037 | updates/centroid=1548.4
  Fill epoch 5/5 | init=100% | spread=0.0051 | updates/centroid=1935.5

[Stage 2.B] Training calibrator on proxy unknowns

Ep    | Loss     | KnAcc   | AUROC   | Recall  | Codebook
-----------------------------------------------------------

In [ ]:
import torch
import platform
import psutil

# 1. GPU Info
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found"
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0

# 2. CPU Info
cpu_info = platform.processor()

# 3. RAM Info
ram_total = psutil.virtual_memory().total / (1024**3)

# 4. PyTorch Version
torch_ver = torch.__version__

print(f"--- THESIS HARDWARE SPECS ---")
print(f"GPU: {gpu_name} ({gpu_mem:.2f} GB VRAM)")
print(f"CPU: {cpu_info} (Intel Xeon)")
print(f"System RAM: {ram_total:.2f} GB")
print(f"PyTorch Version: {torch_ver}")
print(f"-----------------------------")


--- THESIS HARDWARE SPECS ---
GPU: NVIDIA L4 (23.66 GB VRAM)
CPU: x86_64 (Intel Xeon)
System RAM: 52.96 GB
PyTorch Version: 2.10.0+cu128
-----------------------------


In [8]:
# ============================================================
#  BASELINE TRAINING CELL — paste into existing Colab notebook
#  Trains VGG-16, ResNet-18, DenseNet-121 with two-phase
#  fine-tuning. Already-trained checkpoints are skipped.
# ============================================================

from google.colab import drive
from pathlib import Path
import sys, os, json, random
from datetime import datetime, timezone

import torch
import torch.nn as nn
import numpy as np

# ── 1. Mount Drive and set project root ──────────────────────
drive.mount('/content/drive', force_remount=False)   # force_remount=False skips
                                                      # re-mount if already mounted

PROJECT_ROOT = Path('/content/drive/Othercomputers/My Laptop/thesis_project')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

# ── 2. Imports (after path is set) ───────────────────────────
from python.src.legacy_models import (
    LiteratureBaseline_VGG16,
    LiteratureBaseline_ResNet18,
    LiteratureBaseline_DenseNet121,
)
from python.src.dataio import load_artifact
from python.src.preprocessing import build_feature_tensor, split_dataset
from python.src.utils import (
    create_train_loader, create_eval_loader,
    resolve_device, FeatureTensorDataset,
)

# ── 3. Config — edit here if needed ──────────────────────────
SEED          = 55
N_PER_CLASS   = 2500
SPEC_VER      = "v2"
N_EPOCHS      = 50
PHASE1_EPOCHS = 10       # head-only warm-up epochs
LR_PHASE1     = 1e-3     # higher LR for randomly-init'd head
LR_PHASE2     = 1e-4     # lower LR for pretrained backbone layers
WEIGHT_DECAY  = 1e-4
BATCH_SIZE    = 32
NUM_CLASSES   = 10

BASELINE_REGISTRY = {
    "vgg_16":       LiteratureBaseline_VGG16,
    "resnet_18":    LiteratureBaseline_ResNet18,
    "densenet_121": LiteratureBaseline_DenseNet121,
}

# ── 4. Helpers ────────────────────────────────────────────────
def _set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True

def _make_loader(split_tuple, batch_size, shuffle):
    x_stft, x_iq, x_if, y = split_tuple
    ds = FeatureTensorDataset(x_stft, x_iq, x_if, y)
    return create_train_loader(ds, batch_size) if shuffle else create_eval_loader(ds, batch_size)

@torch.no_grad()
def _evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct, total_n = 0.0, 0, 0
    for x_stft, x_iq, x_if, y in loader:
        x_stft, x_iq, x_if, y = x_stft.to(device), x_iq.to(device), x_if.to(device), y.to(device)
        logits = model(x_stft, x_iq, x_if)
        total_loss    += criterion(logits, y).item() * y.size(0)
        total_correct += (logits.argmax(1) == y).sum().item()
        total_n       += y.size(0)
    return total_loss / total_n, total_correct / total_n

def _run_phase(model, train_loader, val_loader, optimizer, scheduler,
               criterion, device, start_ep, end_ep, phase_label, log_epochs):
    best_acc, best_state = 0.0, None
    for epoch in range(start_ep, end_ep + 1):
        model.train()
        total_loss, total_correct, total_n = 0.0, 0, 0
        for x_stft, x_iq, x_if, y in train_loader:
            x_stft, x_iq, x_if, y = x_stft.to(device), x_iq.to(device), x_if.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x_stft, x_iq, x_if)
            loss   = criterion(logits, y)
            loss.backward()
            optimizer.step()
            bs = y.size(0)
            total_loss    += loss.item() * bs
            total_correct += (logits.argmax(1) == y).sum().item()
            total_n       += bs

        train_loss = total_loss / total_n
        train_acc  = total_correct / total_n
        val_loss, val_acc = _evaluate(model, val_loader, criterion, device)
        scheduler.step()
        lr_now = scheduler.get_last_lr()[0]

        if val_acc > best_acc:
            best_acc   = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if epoch == start_ep or epoch % 5 == 0 or epoch == end_ep:
            print(f"    [{phase_label}] Ep {epoch:02d}/{end_ep} | "
                  f"LR {lr_now:.2e} | Train {train_loss:.4f} ({100*train_acc:.1f}%) | "
                  f"Val {val_loss:.4f} ({100*val_acc:.2f}%)")

        log_epochs.append({"epoch": epoch, "phase": phase_label,
                           "train_loss": train_loss, "train_acc": train_acc,
                           "val_loss": val_loss, "val_accuracy": val_acc, "lr": lr_now})
    return best_acc, best_state

# ── 5. Load dataset once (shared across all three models) ────
print("\nLoading dataset...")

_train_path = (PROJECT_ROOT / "artifacts" / "datasets" / "impaired"
               / f"impaired_dataset_{SPEC_VER}_seed{SEED}_n{N_PER_CLASS}_train.mat")
_eval_path  = (PROJECT_ROOT / "artifacts" / "datasets" / "impaired"
               / f"impaired_dataset_{SPEC_VER}_seed{SEED}_n{N_PER_CLASS}_eval.mat")

_train_artifact = load_artifact(str(_train_path), load_params=False)
_x_stft, _x_iq, _x_if, _y = build_feature_tensor(_train_artifact)

# split_dataset returns two tuples — kept as tuples throughout
train_set, val_set = split_dataset(_x_stft, _x_iq, _x_if, _y, train_ratio=0.8, seed=SEED)

# Extract tensors from the Subsets (as you already know works)
x_stft_tr = train_set.dataset.tensors[0][train_set.indices]
x_iq_tr = train_set.dataset.tensors[1][train_set.indices]
x_if_tr = train_set.dataset.tensors[2][train_set.indices]
y_tr = train_set.dataset.tensors[3][train_set.indices]
x_stft_val = val_set.dataset.tensors[0][val_set.indices]
x_iq_val = val_set.dataset.tensors[1][val_set.indices]
x_if_val = val_set.dataset.tensors[2][val_set.indices]
y_val = val_set.dataset.tensors[3][val_set.indices]

train_split = (x_stft_tr, x_iq_tr, x_if_tr, y_tr)
val_split = (x_stft_val, x_iq_val, x_if_val, y_val)

_eval_artifact = load_artifact(str(_eval_path), load_params=False)
_xs, _xi, _xf, _yt = build_feature_tensor(_eval_artifact)
test_data = (_xs, _xi, _xf, _yt)

print(f"  Train : {train_split[0].shape[0]} | Val : {val_split[0].shape[0]} | Test : {test_data[0].shape[0]}")

device = resolve_device("auto")
print(f"  Device: {device}")

# ── 6. Train all baselines ────────────────────────────────────
for model_name, model_cls in BASELINE_REGISTRY.items():

    ckpt_path = (PROJECT_ROOT / "artifacts" / "checkpoints"
                 / f"{model_name}_baseline_seed{SEED}_n{N_PER_CLASS}.pt")

    if ckpt_path.exists():
        print(f"\n[SKIP] {model_name} — checkpoint already exists.")
        continue

    print(f"\n{'='*65}")
    print(f"  Training: {model_name.upper()}")
    print(f"{'='*65}")

    _set_seed(SEED)
    model     = model_cls(num_classes=NUM_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    train_loader = _make_loader(train_split, BATCH_SIZE, shuffle=True)
    val_loader   = _make_loader(val_split,   BATCH_SIZE, shuffle=False)
    test_loader  = _make_loader(test_data,   BATCH_SIZE, shuffle=False)

    log_epochs      = []
    global_best_acc = 0.0
    global_best_state = None

    # Phase 1 — head only
    model.freeze_for_phase1()
    p1_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Phase 1 trainable params : {p1_params:,}")

    opt1 = torch.optim.Adam([p for p in model.parameters() if p.requires_grad],
                             lr=LR_PHASE1, weight_decay=WEIGHT_DECAY)
    sch1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=PHASE1_EPOCHS)

    best1, state1 = _run_phase(model, train_loader, val_loader, opt1, sch1,
                                criterion, device, 1, PHASE1_EPOCHS,
                                "Phase1", log_epochs)
    if best1 > global_best_acc:
        global_best_acc, global_best_state = best1, state1

    # Phase 2 — last block + head
    model.unfreeze_for_phase2()
    p2_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Phase 2 trainable params : {p2_params:,}")

    phase2_len = N_EPOCHS - PHASE1_EPOCHS
    opt2 = torch.optim.Adam([p for p in model.parameters() if p.requires_grad],
                             lr=LR_PHASE2, weight_decay=WEIGHT_DECAY)
    sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=phase2_len)

    best2, state2 = _run_phase(model, train_loader, val_loader, opt2, sch2,
                                criterion, device,
                                PHASE1_EPOCHS + 1, N_EPOCHS,
                                "Phase2", log_epochs)
    if best2 > global_best_acc:
        global_best_acc, global_best_state = best2, state2

    # Final test
    model.load_state_dict(global_best_state)
    test_loss, test_acc = _evaluate(model, test_loader, criterion, device)
    print(f"\n  Best Val Acc : {100*global_best_acc:.2f}%")
    print(f"  Test Acc     : {100*test_acc:.2f}%")

    # Save checkpoint
    ckpt_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(global_best_state, ckpt_path)
    print(f"  Checkpoint   : {ckpt_path.name}")

    # Save log
    log_dir = PROJECT_ROOT / "artifacts" / "logs" / "baselines"
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f"{model_name}_baseline_seed{SEED}_n{N_PER_CLASS}.json"
    with log_path.open("w") as f:
        json.dump({"model": model_name, "seed": SEED, "n_per_class": N_PER_CLASS,
                   "best_val_acc": global_best_acc, "test_acc": test_acc,
                   "test_loss": test_loss, "epochs": log_epochs}, f, indent=2)
    print(f"  Log          : {log_path.name}")

    del model
    torch.cuda.empty_cache()

print("\n\n✓ All baselines complete.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/Othercomputers/My Laptop/thesis_project

Loading dataset...
  Train : 20000 | Val : 5000 | Test : 25000
  Device: cuda

  Training: VGG_16
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 232MB/s]


  Phase 1 trainable params : 40,970
    [Phase1] Ep 01/10 | LR 9.76e-04 | Train 2.4292 (11.5%) | Val 2.2846 (15.20%)
    [Phase1] Ep 05/10 | LR 5.00e-04 | Train 2.3528 (13.2%) | Val 2.2923 (15.08%)
    [Phase1] Ep 10/10 | LR 0.00e+00 | Train 2.2983 (13.7%) | Val 2.2559 (17.12%)
  Phase 2 trainable params : 126,666,250
    [Phase2] Ep 11/50 | LR 9.98e-05 | Train 2.1523 (20.3%) | Val 2.0469 (28.18%)
    [Phase2] Ep 15/50 | LR 9.62e-05 | Train 1.5973 (50.3%) | Val 1.8235 (39.16%)
    [Phase2] Ep 20/50 | LR 8.54e-05 | Train 0.5687 (99.9%) | Val 2.0656 (39.10%)
    [Phase2] Ep 25/50 | LR 6.91e-05 | Train 0.5409 (100.0%) | Val 2.0108 (39.02%)
    [Phase2] Ep 30/50 | LR 5.00e-05 | Train 0.5203 (100.0%) | Val 1.9550 (39.34%)
    [Phase2] Ep 35/50 | LR 3.09e-05 | Train 0.5125 (100.0%) | Val 1.9339 (39.96%)
    [Phase2] Ep 40/50 | LR 1.46e-05 | Train 0.5082 (100.0%) | Val 1.9165 (39.88%)
    [Phase2] Ep 45/50 | LR 3.81e-06 | Train 0.5060 (100.0%) | Val 1.9116 (40.06%)
    [Phase2] Ep 50/50 | LR 

100%|██████████| 44.7M/44.7M [00:00<00:00, 142MB/s]


  Phase 1 trainable params : 5,130
    [Phase1] Ep 01/10 | LR 9.76e-04 | Train 2.3508 (14.3%) | Val 2.2667 (19.10%)
    [Phase1] Ep 05/10 | LR 5.00e-04 | Train 2.1089 (26.2%) | Val 2.1636 (24.74%)
    [Phase1] Ep 10/10 | LR 0.00e+00 | Train 2.0577 (29.5%) | Val 2.1167 (25.48%)
  Phase 2 trainable params : 8,398,858
    [Phase2] Ep 11/50 | LR 9.98e-05 | Train 1.7932 (41.4%) | Val 1.6523 (46.96%)
    [Phase2] Ep 15/50 | LR 9.62e-05 | Train 0.5660 (100.0%) | Val 1.8529 (46.40%)
    [Phase2] Ep 20/50 | LR 8.54e-05 | Train 0.5600 (100.0%) | Val 1.8127 (47.04%)
    [Phase2] Ep 25/50 | LR 6.91e-05 | Train 0.5345 (100.0%) | Val 1.7285 (48.82%)
    [Phase2] Ep 30/50 | LR 5.00e-05 | Train 0.5243 (100.0%) | Val 1.6971 (49.58%)
    [Phase2] Ep 35/50 | LR 3.09e-05 | Train 0.5166 (100.0%) | Val 1.6793 (50.04%)
    [Phase2] Ep 40/50 | LR 1.46e-05 | Train 0.5118 (100.0%) | Val 1.6683 (50.14%)
    [Phase2] Ep 45/50 | LR 3.81e-06 | Train 0.5095 (100.0%) | Val 1.6708 (50.06%)
    [Phase2] Ep 50/50 | LR 0

100%|██████████| 30.8M/30.8M [00:00<00:00, 166MB/s]


  Phase 1 trainable params : 10,250
    [Phase1] Ep 01/10 | LR 9.76e-04 | Train 2.2864 (16.8%) | Val 2.2161 (20.78%)
    [Phase1] Ep 05/10 | LR 5.00e-04 | Train 2.0447 (29.4%) | Val 2.0821 (26.00%)
    [Phase1] Ep 10/10 | LR 0.00e+00 | Train 1.9914 (32.4%) | Val 2.0597 (27.72%)
  Phase 2 trainable params : 2,170,378
    [Phase2] Ep 11/50 | LR 9.98e-05 | Train 1.8975 (36.5%) | Val 1.8602 (36.22%)
    [Phase2] Ep 15/50 | LR 9.62e-05 | Train 0.6929 (98.0%) | Val 2.1528 (37.24%)
    [Phase2] Ep 20/50 | LR 8.54e-05 | Train 0.5659 (100.0%) | Val 2.1097 (37.04%)
    [Phase2] Ep 25/50 | LR 6.91e-05 | Train 0.5482 (100.0%) | Val 2.0422 (37.78%)
    [Phase2] Ep 30/50 | LR 5.00e-05 | Train 0.5345 (100.0%) | Val 1.9898 (39.26%)
    [Phase2] Ep 35/50 | LR 3.09e-05 | Train 0.5272 (100.0%) | Val 1.9757 (39.08%)
    [Phase2] Ep 40/50 | LR 1.46e-05 | Train 0.5212 (100.0%) | Val 1.9580 (39.18%)
    [Phase2] Ep 45/50 | LR 3.81e-06 | Train 0.5188 (100.0%) | Val 1.9560 (39.48%)
    [Phase2] Ep 50/50 | LR 0